# Project 04: Retail Sales Forecasting Engine
### Time-Series Feature Engineering & Regression Forecasting

Predicting next-period store sales using autoregressive lag variables and rolling window averages.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_percentage_error

# Generate synthetic daily sales data for 365 days
dates = pd.date_range(start="2023-01-01", periods=365, freq="D")
trend = np.linspace(5000, 8000, 365)
seasonality = 1200 * np.sin(np.linspace(0, 2 * np.pi * 52, 365))
noise = np.random.normal(0, 300, 365)
sales = trend + seasonality + noise

df_sales = pd.DataFrame({"Date": dates, "Sales": sales})
# Create lag features
df_sales["Lag_1"] = df_sales["Sales"].shift(1)
df_sales["Lag_7"] = df_sales["Sales"].shift(7)
df_sales["Rolling_Mean_7"] = df_sales["Sales"].shift(1).rolling(7).mean()
df_sales = df_sales.dropna()

train_df = df_sales.iloc[:-30]
test_df = df_sales.iloc[-30:]

X_cols = ["Lag_1", "Lag_7", "Rolling_Mean_7"]
model = GradientBoostingRegressor(n_estimators=100, random_state=42)
model.fit(train_df[X_cols], train_df["Sales"])

preds = model.predict(test_df[X_cols])
mape = mean_absolute_percentage_error(test_df["Sales"], preds)
print(f"30-Day Sales Forecast MAPE: {mape:.2%}")
print("Sample Next 5 Days Forecast vs Actual:")
for act, pr in zip(test_df["Sales"].values[:5], preds[:5]):
    print(f"Actual: ${act:,.2f} | Forecast: ${pr:,.2f}")